# 13 — 5-Flavor HVG Validation Results

> **Note on R² semantics (added 2026-05-06).** The upstream `cellot/cellot_gpu/scripts/evaluate.py` writes a metric labeled `r2-means` (also `r2-stds`, `r2-pairwise_feat_corrs`) into every `evals.csv`, but the underlying call is `pd.Series.corr(...)` which returns the Pearson correlation coefficient `r ∈ [-1, 1]` — NOT R². Earlier revisions of this notebook consumed those values as if they were R², which silently inflated every reported R² number. This notebook now squares all r2-* metric values immediately after loading each `evals.csv`, so every pivot table, heatmap, and scatter plot reports **true R² (coefficient of determination)**. Compare with `08.1_renorm_vs_stale_comparison.ipynb`, which already did this conversion.

Idempotent results aggregator + figure generator for the 5-HVG-flavor x 4-group x 2-mode x 2-model matrix.

This notebook walks `cellot/cellot_gpu/results/hvg_*/` and gathers every `evals.csv` it finds (under `evals_ood_latent_space/`). It does NOT require all 80 evaluations to be present — it shows whichever cells have completed and reports missing cells.

**Outputs (per re-run, written under `hvg_flavor_results_outputs/`):**
- `results_long.csv` — every `(flavor, group, mode, model, ncells, metric, value)` row from every available `evals.csv`.
- `results_pivot_R2_means.csv` — the headline pivot table over (flavor, group, mode, model).
- `results_pivot_MMD.csv` — same for MMD.
- `figures/r2_means_heatmap_per_flavor.{pdf,png}` — 4 small per-flavor heatmaps stacked into one figure: rows = (group, mode), cols = (scGen, IMPACT_CellOT), cell value = R² of means.
- `figures/mmd_heatmap_per_flavor.{pdf,png}` — same layout, cell value = MMD.
- `figures/method_gap_vs_distribution_gap.{pdf,png}` — scatter showing how much OT helps (method gap) vs how much OOD hurts (distribution gap), one point per (flavor, group).
- `figures/biomarker_density_PTPRC_CD3E.{pdf,png}` — for the (flavor, group, mode) cell with the best IMPACT R², per-gene density of `PTPRC` and `CD3E` in actual-target vs IMPACT-predicted vs scGen-predicted (data-space; recomputed from `imputed.h5ad` if available).

**Status table:** prints how many of the 80 expected (flavor, group, mode, model) eval cells exist at the time of the run, and which are still missing.

Re-run this notebook anytime more `evals.csv` files land — figures and CSVs will be regenerated from whatever is currently on disk.

In [12]:
import os
import re
import json
import warnings
from pathlib import Path
from itertools import product

import matplotlib
matplotlib.use("Agg")  # ensure savefig works in nbconvert without display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

warnings.filterwarnings("ignore")

BASE = Path("/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT")
RESULTS_DIR = BASE / "cellot/cellot_gpu/results"
OUT_DIR = BASE / "speciesOT/baseline/analysis/hvg_flavor_results_outputs"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

FLAVORS = ["seurat", "cell_ranger", "seurat_v3", "seurat_v3_paper", "pearson_residuals"]
GROUPS = ["a", "b", "c", "d"]
GROUP_NAMES = {"a": "cd8", "b": "cd8_thymo", "c": "tcell_subtypes", "d": "cd4"}
MODES = ["ood", "iid"]
MODELS = ["scgen", "impact_cellot"]
MODEL_LABEL = {"scgen": "scGen", "impact_cellot": "IMPACT_CellOT"}

EVAL_SUBDIR = "evals_ood_latent_space"

print("results dir:", RESULTS_DIR)
print("outputs   :", OUT_DIR)
print("expecting :", len(FLAVORS) * len(GROUPS) * len(MODES) * len(MODELS), "cells")

results dir: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/results
outputs   : /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/hvg_flavor_results_outputs
expecting : 80 cells


## 1. Discover available eval csvs and report status

In [13]:
def cell_eval_path(flavor, gk, mode, model):
    return RESULTS_DIR / f"hvg_{flavor}_{gk}_{mode}" / model / EVAL_SUBDIR / "evals.csv"

def cell_imputed_path(flavor, gk, mode, model):
    return RESULTS_DIR / f"hvg_{flavor}_{gk}_{mode}" / model / EVAL_SUBDIR / "imputed.h5ad"

def cell_train_status_path(flavor, gk, mode, model):
    return RESULTS_DIR / f"hvg_{flavor}_{gk}_{mode}" / model / "cache" / "status"

status_rows = []
for flavor, gk, mode, model in product(FLAVORS, GROUPS, MODES, MODELS):
    eval_csv = cell_eval_path(flavor, gk, mode, model)
    has_eval = eval_csv.exists() and eval_csv.stat().st_size > 200  # > header only
    train_status = cell_train_status_path(flavor, gk, mode, model)
    train_state = "missing"
    if train_status.exists():
        try:
            train_state = train_status.read_text().strip()
        except Exception:
            train_state = "unreadable"
    status_rows.append({
        "flavor": flavor, "group": gk, "mode": mode, "model": model,
        "trained": train_state, "evaluated": bool(has_eval),
        "eval_csv": str(eval_csv) if has_eval else "",
    })

status_df = pd.DataFrame(status_rows)
status_df.to_csv(OUT_DIR / "status.csv", index=False)

n_total = len(status_df)
n_trained = (status_df["trained"] == "done").sum()
n_evaluated = status_df["evaluated"].sum()
print(f"Status: {n_trained}/{n_total} trainings done; {n_evaluated}/{n_total} evals done")
print()
status_pivot = (
    status_df
    .assign(state=lambda d: np.where(d["evaluated"], "EVAL", np.where(d["trained"] == "done", "TRAIN", d["trained"])))
    .pivot_table(index=["flavor", "group", "mode"], columns="model", values="state", aggfunc="first")
    .fillna("--")
    .reindex(columns=MODELS)
)
print(status_pivot.to_string())

Status: 80/80 trainings done; 80/80 evals done

model                        scgen impact_cellot
flavor            group mode                    
cell_ranger       a     iid   EVAL          EVAL
                        ood   EVAL          EVAL
                  b     iid   EVAL          EVAL
                        ood   EVAL          EVAL
                  c     iid   EVAL          EVAL
                        ood   EVAL          EVAL
                  d     iid   EVAL          EVAL
                        ood   EVAL          EVAL
pearson_residuals a     iid   EVAL          EVAL
                        ood   EVAL          EVAL
                  b     iid   EVAL          EVAL
                        ood   EVAL          EVAL
                  c     iid   EVAL          EVAL
                        ood   EVAL          EVAL
                  d     iid   EVAL          EVAL
                        ood   EVAL          EVAL
seurat            a     iid   EVAL          EVAL
                     

## 2. Aggregate every available `evals.csv`

`evaluate.py` writes a long-format CSV with columns `[ncells, nfeatures, metric, value]`. We average over `nreps` (already aggregated by ncells) and pick the largest `ncells` value present per cell as the headline number for that cell.

In [14]:
R2_METRICS = {"r2-means", "r2-stds", "r2-pairwise_feat_corrs"}

long_rows = []
for _, r in status_df[status_df["evaluated"]].iterrows():
    df = pd.read_csv(r["eval_csv"])
    # evaluate.py mislabels Pearson r as r2-* — square here to get true R^2
    is_r2 = df["metric"].isin(R2_METRICS)
    df.loc[is_r2, "value"] = df.loc[is_r2, "value"] ** 2
    df["flavor"] = r["flavor"]
    df["group"] = r["group"]
    df["mode"] = r["mode"]
    df["model"] = r["model"]
    long_rows.append(df)

if not long_rows:
    print("NO eval csvs available yet. Re-run this notebook after some evaluations land.")
    long_df = pd.DataFrame(columns=[
        "ncells", "nfeatures", "metric", "value",
        "flavor", "group", "mode", "model",
    ])
else:
    long_df = pd.concat(long_rows, ignore_index=True)

long_df.to_csv(OUT_DIR / "results_long.csv", index=False)
print(f"results_long: {len(long_df)} rows ({long_df['flavor'].nunique() if len(long_df) else 0} flavors, {long_df['group'].nunique() if len(long_df) else 0} groups)")
long_df.head()

results_long: 17280 rows (5 flavors, 4 groups)


,ncells,nfeatures,metric,value,flavor,group,mode,model
0,30,all,l2-means,7.640812,seurat,a,ood,scgen
1,30,all,l2-stds,2.051189,seurat,a,ood,scgen
2,30,all,r2-means,0.685781,seurat,a,ood,scgen
3,30,all,r2-stds,0.179380,seurat,a,ood,scgen
4,30,all,r2-pairwise_feat_corrs,0.066076,seurat,a,ood,scgen


## 3. Headline pivots: R² of means and MMD

For each `(flavor, group, mode, model)` cell, take the metric value at the largest `ncells` reported (averaging over nreps already happened upstream by the eval iterator). This gives the most stable, large-sample estimate per cell.

In [15]:
def headline(metric):
    if long_df.empty:
        return pd.DataFrame()
    sub = long_df[long_df["metric"] == metric].copy()
    if sub.empty:
        return pd.DataFrame()
    # average over nreps for each (flavor, group, mode, model, ncells)
    agg = (
        sub.groupby(["flavor", "group", "mode", "model", "ncells"], as_index=False)["value"]
           .mean()
    )
    # take the largest ncells value as the representative for that cell
    largest = agg.sort_values("ncells").drop_duplicates(
        ["flavor", "group", "mode", "model"], keep="last"
    )
    pivot = largest.pivot_table(
        index=["flavor", "group", "mode"], columns="model", values="value"
    ).reindex(columns=MODELS)
    return pivot

r2_pivot = headline("r2-means")
mmd_pivot = headline("mmd")
r2_stds_pivot = headline("r2-stds")
enrich_pivot = headline("enrichment-k50")

r2_pivot.to_csv(OUT_DIR / "results_pivot_R2_means.csv")
mmd_pivot.to_csv(OUT_DIR / "results_pivot_MMD.csv")
r2_stds_pivot.to_csv(OUT_DIR / "results_pivot_R2_stds.csv")
enrich_pivot.to_csv(OUT_DIR / "results_pivot_enrichment_k50.csv")

print("R^2 of means (per flavor x group x mode x model)")
print("=" * 70)
print(r2_pivot.round(4).to_string())
print()
print("MMD")
print("=" * 70)
print(mmd_pivot.round(4).to_string())

R^2 of means (per flavor x group x mode x model)
model                          scgen  impact_cellot
flavor            group mode                       
cell_ranger       a     iid   0.6782         0.7373
                        ood   0.7168         0.7632
                  b     iid   0.6950         0.8377
                        ood   0.5731         0.5038
                  c     iid   0.7189         0.8575
                        ood   0.7209         0.6155
                  d     iid   0.6891         0.7940
                        ood   0.8061         0.8377
pearson_residuals a     iid   0.6116         0.7610
                        ood   0.6783         0.7829
                  b     iid   0.7025         0.8441
                        ood   0.7276         0.7972
                  c     iid   0.7794         0.8592
                        ood   0.7571         0.8255
                  d     iid   0.6870         0.7524
                        ood   0.7336         0.7894
seurat         

## 4. Per-flavor heatmap of R² and MMD

One subplot per flavor. Rows = (group, mode), columns = (scGen, IMPACT_CellOT). Color encodes the metric value. Missing cells appear as gray.

In [16]:
def plot_per_flavor_heatmaps(pivot, metric_label, cmap, vmin=None, vmax=None,
                              fmt="{:.3f}", out_stem=None):
    if pivot.empty:
        print(f"  no data for {metric_label}; skipping plot")
        return
    rows = [(gk, mode) for gk in GROUPS for mode in MODES]
    row_labels = [f"{gk.upper()} {mode.upper()}" for gk, mode in rows]

    n = len(FLAVORS)
    fig, axes = plt.subplots(1, n, figsize=(2.6 * n, 4.2), sharey=True)
    if n == 1:
        axes = [axes]

    if vmin is None:
        vmin = float(pivot.min().min()) if pivot.notna().any().any() else 0.0
    if vmax is None:
        vmax = float(pivot.max().max()) if pivot.notna().any().any() else 1.0
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    for ax, flavor in zip(axes, FLAVORS):
        # rows: (group, mode), cols: model
        if flavor in pivot.index.get_level_values(0):
            sub = pivot.xs(flavor, level="flavor", drop_level=True)
            # build a (8 rows x 2 cols) matrix in the canonical order
            mat = np.full((len(rows), len(MODELS)), np.nan)
            for i, (gk, mode) in enumerate(rows):
                for j, model in enumerate(MODELS):
                    if (gk, mode) in sub.index and model in sub.columns:
                        v = sub.loc[(gk, mode), model]
                        if pd.notna(v):
                            mat[i, j] = v
        else:
            mat = np.full((len(rows), len(MODELS)), np.nan)

        ax.imshow(mat, cmap=cmap, norm=norm, aspect="auto")
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                v = mat[i, j]
                txt = fmt.format(v) if pd.notna(v) else "—"
                color = "white" if (pd.notna(v) and norm(v) > 0.55) else "black"
                ax.text(j, i, txt, ha="center", va="center", color=color, fontsize=8)
        ax.set_xticks(range(len(MODELS)))
        ax.set_xticklabels([MODEL_LABEL[m] for m in MODELS], fontsize=9, rotation=15)
        ax.set_yticks(range(len(rows)))
        ax.set_yticklabels(row_labels, fontsize=8)
        ax.set_title(flavor, fontsize=10)

    fig.suptitle(metric_label, fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.95])

    if out_stem:
        for ext in ("pdf", "png"):
            fig.savefig(FIG_DIR / f"{out_stem}.{ext}", dpi=150, bbox_inches="tight")
        print(f"  saved {out_stem}.{{pdf,png}}")
    plt.show()
    plt.close(fig)


plot_per_flavor_heatmaps(
    r2_pivot, "R² of means (latent space)", cmap="viridis",
    vmin=0.0, vmax=1.0, out_stem="r2_means_heatmap_per_flavor",
)
plot_per_flavor_heatmaps(
    mmd_pivot, "MMD (latent space)  — lower is better", cmap="viridis_r",
    fmt="{:.4f}", out_stem="mmd_heatmap_per_flavor",
)

  saved r2_means_heatmap_per_flavor.{pdf,png}
  saved mmd_heatmap_per_flavor.{pdf,png}


## 5. Method gap vs distribution gap scatter

Per `(flavor, group)`, plot:
- `method_gap_OOD`  = R²(IMPACT, OOD) − R²(scGen, OOD) on the y-axis. Positive means OT helps OOD.
- `distribution_gap_IMPACT` = R²(IMPACT, IID) − R²(IMPACT, OOD) on the x-axis. Positive means IID is easier for IMPACT than OOD.

Color = flavor; marker = group. The interesting region is the upper-left: low distribution gap (the model isn't getting much from seeing IID half) AND high method gap (OT is rescuing OOD even when scGen alone fails). That's the "Pearson HVG + IMPACT" success story.

In [17]:
def gap_scatter(pivot, metric_label, out_stem, lower_is_better=False):
    if pivot.empty:
        print(f"  no data for gap scatter ({metric_label}); skipping")
        return
    sign = -1 if lower_is_better else 1
    rows = []
    for flavor in FLAVORS:
        if flavor not in pivot.index.get_level_values(0):
            continue
        sub = pivot.xs(flavor, level="flavor", drop_level=True)
        for gk in GROUPS:
            try:
                ood_scgen = sub.loc[(gk, "ood"), "scgen"]
                ood_impact = sub.loc[(gk, "ood"), "impact_cellot"]
                iid_scgen = sub.loc[(gk, "iid"), "scgen"]
                iid_impact = sub.loc[(gk, "iid"), "impact_cellot"]
            except KeyError:
                continue
            if any(pd.isna(v) for v in (ood_scgen, ood_impact, iid_scgen, iid_impact)):
                continue
            rows.append({
                "flavor": flavor, "group": gk,
                "method_gap_OOD":   sign * (ood_impact - ood_scgen),
                "method_gap_IID":   sign * (iid_impact - iid_scgen),
                "dist_gap_scgen":   sign * (iid_scgen - ood_scgen),
                "dist_gap_impact":  sign * (iid_impact - ood_impact),
            })
    if not rows:
        print(f"  no complete (flavor, group) cells for {metric_label}; skipping")
        return None
    gap_df = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(6.5, 5))
    color_map = {f: c for f, c in zip(FLAVORS, plt.cm.tab10(np.linspace(0, 1, len(FLAVORS))))}
    marker_map = {"a": "o", "b": "s", "c": "^", "d": "D"}

    for _, r in gap_df.iterrows():
        ax.scatter(
            r["dist_gap_impact"], r["method_gap_OOD"],
            c=[color_map[r["flavor"]]], marker=marker_map[r["group"]],
            s=120, edgecolors="black", linewidths=0.5,
        )
        ax.annotate(
            f"{r['flavor'][:3]}-{r['group'].upper()}",
            (r["dist_gap_impact"], r["method_gap_OOD"]),
            fontsize=7, alpha=0.8, xytext=(4, 4), textcoords="offset points",
        )
    ax.axhline(0, color="grey", lw=0.5)
    ax.axvline(0, color="grey", lw=0.5)
    ax.set_xlabel("distribution gap (IMPACT, IID − OOD)" + ("  [neg = OOD better, unusual]" if not lower_is_better else ""))
    ax.set_ylabel(f"method gap OOD ({metric_label})\nIMPACT − scGen")
    ax.set_title(f"{metric_label}: how much does OT help, vs how much does OOD hurt?")

    handles = [plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=color_map[f], markersize=8, label=f) for f in FLAVORS]
    handles += [plt.Line2D([0], [0], marker=marker_map[g], color="grey", markersize=8, lw=0, label=f"Group {g.upper()}") for g in GROUPS]
    ax.legend(handles=handles, fontsize=7, loc="best", ncol=2)

    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"{out_stem}.{ext}", dpi=150, bbox_inches="tight")
    print(f"  saved {out_stem}.{{pdf,png}}")
    plt.show()
    plt.close(fig)
    gap_df.to_csv(OUT_DIR / f"{out_stem}.csv", index=False)
    return gap_df


gap_r2 = gap_scatter(r2_pivot, "R²", "method_gap_vs_distribution_gap_R2")
gap_mmd = gap_scatter(mmd_pivot, "MMD", "method_gap_vs_distribution_gap_MMD", lower_is_better=True)

  saved method_gap_vs_distribution_gap_R2.{pdf,png}
  saved method_gap_vs_distribution_gap_MMD.{pdf,png}


## 6. Biomarker density plots (PTPRC, CD3E)

Pull the `imputed.h5ad` from a chosen `(flavor, group, mode)` cell, plus its source dataset for the actual-target reference. Plot per-gene density of PTPRC and CD3E for: actual mouse (source), actual human (target), scGen prediction, IMPACT prediction.

Caveat: this requires `--where data_space` evaluations. The matrix is currently set up for `--where latent_space` (latent-space metrics in the heatmaps above). For data-space density plots a separate eval pass with `--where data_space` is needed. If those imputed.h5ad files don't exist yet, this cell will print a notice and skip.

Also requires the gene to have been selected as HVG by the chosen flavor (PTPRC by Pearson; PTPRC and CD3E by all flavors per `01.4` summary).

In [18]:
try:
    import anndata as ad
except ImportError:
    ad = None

MARKERS = {"PTPRC (CD45)": "ENSG00000081237", "CD3E": "ENSG00000198851"}

DATA_DIR = BASE / "cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg"

# pick the (flavor, group, mode) cell with the best IMPACT R^2 to feature
if not r2_pivot.empty and "impact_cellot" in r2_pivot.columns:
    impact_only = r2_pivot["impact_cellot"].dropna()
    if len(impact_only):
        best_key = impact_only.idxmax()
        best_flavor, best_gk, best_mode = best_key
        best_r2 = float(impact_only.max())
        print(f"Best IMPACT cell so far: {best_flavor} / Group {best_gk.upper()} / {best_mode.upper()}  R^2={best_r2:.4f}")
    else:
        best_flavor = best_gk = best_mode = None
        print("No IMPACT R^2 numbers yet; skipping biomarker plot.")
else:
    best_flavor = best_gk = best_mode = None
    print("Pivot table empty; skipping biomarker plot.")


def load_imputed_for_cell(flavor, gk, mode, model):
    p = cell_imputed_path(flavor, gk, mode, model)
    if not p.exists() or ad is None:
        return None
    try:
        return ad.read_h5ad(p)
    except Exception as e:
        print(f"  could not read {p}: {e}")
        return None


def biomarker_density_plot(flavor, gk, mode, out_stem):
    if flavor is None or ad is None:
        return None
    impact_a = load_imputed_for_cell(flavor, gk, mode, "impact_cellot")
    scgen_a = load_imputed_for_cell(flavor, gk, mode, "scgen")
    if impact_a is None and scgen_a is None:
        print(f"  no imputed.h5ad for {flavor}/{gk}/{mode}; skipping density plot.")
        return None
    src_path = DATA_DIR / f"hvg_{flavor}_{gk}_v07.h5ad"
    if not src_path.exists():
        print(f"  source dataset {src_path} missing; skipping.")
        return None
    src = ad.read_h5ad(src_path)
    holdout_lut = {
        "a": ["CL:0000625"],
        "b": ["CL:0000625", "CL:0000893"],
        "c": ["CL:0000624", "CL:0000625", "CL:0000893"],
        "d": ["CL:0000624"],
    }
    is_holdout = src.obs["cell_type_ontology_term_id"].astype(str).isin(holdout_lut[gk])
    actual_mouse = src[(src.obs["condition"] == "mouse") & is_holdout]
    actual_human = src[(src.obs["condition"] == "human") & is_holdout]

    fig, axes = plt.subplots(1, len(MARKERS), figsize=(5.5 * len(MARKERS), 4))
    if len(MARKERS) == 1:
        axes = [axes]
    for ax, (label, ensg) in zip(axes, MARKERS.items()):
        if ensg not in src.var_names:
            ax.text(0.5, 0.5, f"{label}\nnot in {flavor}\ntop-1000",
                    ha="center", va="center", transform=ax.transAxes)
            ax.set_xticks([]); ax.set_yticks([])
            continue
        gi = list(src.var_names).index(ensg)
        traces = []
        if len(actual_mouse) > 0:
            traces.append(("actual mouse", np.asarray(actual_mouse.X[:, gi]).ravel(), "tab:blue"))
        if len(actual_human) > 0:
            traces.append(("actual human", np.asarray(actual_human.X[:, gi]).ravel(), "tab:green"))
        if scgen_a is not None and ensg in scgen_a.var_names:
            gj = list(scgen_a.var_names).index(ensg)
            traces.append(("scGen pred",   np.asarray(scgen_a.X[:, gj]).ravel(), "tab:orange"))
        if impact_a is not None and ensg in impact_a.var_names:
            gj = list(impact_a.var_names).index(ensg)
            traces.append(("IMPACT pred", np.asarray(impact_a.X[:, gj]).ravel(), "tab:red"))
        for name, vals, color in traces:
            if len(vals) == 0:
                continue
            ax.hist(vals, bins=40, density=True, alpha=0.4, color=color, label=name)
        ax.set_title(f"{label} ({ensg})")
        ax.legend(fontsize=8)
        ax.set_xlabel("expression (latent or log-norm)")
        ax.set_ylabel("density")
    fig.suptitle(f"{flavor} / Group {gk.upper()} / {mode.upper()}")
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    for ext in ("pdf", "png"):
        fig.savefig(FIG_DIR / f"{out_stem}.{ext}", dpi=150, bbox_inches="tight")
    print(f"  saved {out_stem}.{{pdf,png}}")
    plt.show()
    plt.close(fig)


if best_flavor is not None:
    biomarker_density_plot(best_flavor, best_gk, best_mode, "biomarker_density_PTPRC_CD3E")

Best IMPACT cell so far: seurat_v3 / Group D / OOD  R^2=0.9184
  saved biomarker_density_PTPRC_CD3E.{pdf,png}


## 7. Biomarker selection matrix across flavors

For each curated biomarker, check whether it landed in the per-(flavor, group) top-1000 HVG list that the new 80-experiment pipeline trained on. The data source for each `(flavor, group)` is the actual dataset file consumed by the pipeline at `cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg/hvg_{flavor}_{gk}_v07.h5ad` — that file is already gene-subsetted to the 1000 HVGs selected by the per-(flavor, group) HVG call in `01.5`, so a marker being in `var_names` is exactly equivalent to being selected.

This complements section 6 (which shows the predicted-vs-actual density for two specific markers in the best-IMPACT cell): here we get a one-glance view of which biomarkers survive each HVG flavor's selection, across all 4 holdout groups, all flavors. The "1/5 — 5/5" column on the right tells you how universal each (marker, group) selection is.

Output: `hvg_flavor_results_outputs/figures/biomarker_selection_matrix.{pdf,png}` and a CSV of the same data.

In [19]:
try:
    import anndata as _ad
except ImportError:
    _ad = None

# Curated biomarker panel for cross-species T/B/myeloid lineage discussion.
# Edit here to add/remove markers.
BIOMARKER_PANEL = {
    "PTPRC (CD45)":  "ENSG00000081237",
    "CD3E":          "ENSG00000198851",
    "CD4":           "ENSG00000010610",
    "CD8A":          "ENSG00000153563",
    "CD5":           "ENSG00000110448",
    "CD7":           "ENSG00000173762",
    "CCR7":          "ENSG00000126353",
    "NCAM1 (CD56)":  "ENSG00000149294",
    "MS4A1 (CD20)":  "ENSG00000156738",
    "CD19":          "ENSG00000177455",
    "CD14":          "ENSG00000170458",
    "ITGAM (CD11b)": "ENSG00000169896",
}

DATA_DIR = BASE / "cellot/cellot_gpu/datasets/speciesot-human-mouse-hvg"

# Load per-(flavor, group) gene sets directly from the dataset .h5ad files
# (each file's var_names is exactly the per-(flavor, group) top-1000 HVG list).
_flavor_group_genes = {}
for flavor in FLAVORS:
    _flavor_group_genes[flavor] = {}
    for gk in GROUPS:
        path = DATA_DIR / f"hvg_{flavor}_{gk}_v07.h5ad"
        if not path.exists() or _ad is None:
            _flavor_group_genes[flavor][gk] = None
        else:
            a = _ad.read_h5ad(path)
            _flavor_group_genes[flavor][gk] = set(a.var_names.astype(str).tolist())

# Build long-form presence DataFrame
_rows = []
for label, ensg in BIOMARKER_PANEL.items():
    for gk in GROUPS:
        row = {"marker": label, "ensg": ensg, "group": gk.upper()}
        for flavor in FLAVORS:
            gset = _flavor_group_genes[flavor].get(gk)
            row[flavor] = "selected" if (gset is not None and ensg in gset) else (
                "missing" if gset is None else "not_selected"
            )
        n_sel = sum(1 for f in FLAVORS
                    if _flavor_group_genes[f].get(gk) and ensg in _flavor_group_genes[f][gk])
        row["n_flavors_selected"] = n_sel
        _rows.append(row)

biomarker_df = pd.DataFrame(_rows)
biomarker_df.to_csv(OUT_DIR / "biomarker_selection_matrix.csv", index=False)

print(f"Saved biomarker_selection_matrix.csv ({len(biomarker_df)} rows = {len(BIOMARKER_PANEL)} markers x {len(GROUPS)} groups)")
print()
print("Compact text view (✓ = in flavor's top-1000):")
print()
short = {"seurat":"seurat", "cell_ranger":"cellrng", "seurat_v3":"v3",
         "seurat_v3_paper":"v3_pap", "pearson_residuals":"pearson"}
header = f"{'marker':<14} {'group':<6}  " + "  ".join(f"{short[f]:<8}" for f in FLAVORS) + "  any"
print(header)
print("-" * len(header))
for label in BIOMARKER_PANEL:
    for gk in GROUPS:
        row = biomarker_df[(biomarker_df["marker"]==label) & (biomarker_df["group"]==gk.upper())].iloc[0]
        marks = ["✓       " if row[f] == "selected" else
                 "?       " if row[f] == "missing" else
                 "—       "
                 for f in FLAVORS]
        print(f"{label:<14} {gk.upper():<6}  " + "  ".join(marks) + f"  {int(row['n_flavors_selected'])}/5")
print()
print("=== Per-flavor recall (across 12 markers x 4 groups = 48 cells) ===")
for f in FLAVORS:
    n_sel = (biomarker_df[f] == "selected").sum()
    print(f"  {f:<22}: {n_sel:>2} / 48 = {n_sel/48:.1%}")

Saved biomarker_selection_matrix.csv (48 rows = 12 markers x 4 groups)

Compact text view (✓ = in flavor's top-1000):

marker         group   seurat    cellrng   v3        v3_pap    pearson   any
----------------------------------------------------------------------------
PTPRC (CD45)   A       —         —         —         —         ✓         1/5
PTPRC (CD45)   B       —         —         —         —         ✓         1/5
PTPRC (CD45)   C       —         —         —         —         ✓         1/5
PTPRC (CD45)   D       —         —         —         —         ✓         1/5
CD3E           A       ✓         ✓         —         —         ✓         3/5
CD3E           B       ✓         ✓         —         —         —         2/5
CD3E           C       ✓         ✓         —         —         —         2/5
CD3E           D       ✓         ✓         —         —         ✓         3/5
CD4            A       —         ✓         —         —         ✓         2/5
CD4            B       —         ✓

In [20]:
def plot_biomarker_selection_heatmap(biomarker_df, out_stem):
    """Heatmap with rows = (marker, group) and cols = flavor. Cell = selected/not-selected."""
    if biomarker_df.empty:
        print("biomarker_df empty; skipping heatmap")
        return
    # Build a numerical matrix: 1 = selected, 0 = not selected, NaN = missing
    val_map = {"selected": 1.0, "not_selected": 0.0, "missing": np.nan}
    n_markers = len(BIOMARKER_PANEL)
    n_groups = len(GROUPS)
    n_rows = n_markers * n_groups
    mat = np.zeros((n_rows, len(FLAVORS)))
    row_labels = []
    i = 0
    for label in BIOMARKER_PANEL:
        for gk in GROUPS:
            sub = biomarker_df[(biomarker_df["marker"]==label) & (biomarker_df["group"]==gk.upper())].iloc[0]
            for j, f in enumerate(FLAVORS):
                mat[i, j] = val_map.get(sub[f], np.nan)
            row_labels.append(f"{label}  ({gk.upper()})")
            i += 1

    cmap = mcolors.ListedColormap(["#d3d3d3", "#2ca02c"])  # gray / green
    bounds = [-0.5, 0.5, 1.5]
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    fig, ax = plt.subplots(figsize=(0.95 * len(FLAVORS) + 3.0, 0.32 * n_rows + 1.2))
    im = ax.imshow(mat, cmap=cmap, norm=norm, aspect="auto")
    # marker text labels in cells
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            if np.isnan(v):
                txt = "?"; col = "black"
            elif v >= 0.5:
                txt = "✓"; col = "white"
            else:
                txt = ""; col = "black"
            if txt:
                ax.text(j, i, txt, ha="center", va="center", color=col, fontsize=9, fontweight="bold")

    ax.set_xticks(range(len(FLAVORS)))
    ax.set_xticklabels(FLAVORS, rotation=30, ha="right", fontsize=10)
    ax.set_yticks(range(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=8)

    # group separators (horizontal lines every 4 rows)
    for k in range(1, n_markers):
        ax.axhline(k * n_groups - 0.5, color="black", lw=0.4, alpha=0.3)

    ax.set_title(f"Biomarker selection across HVG flavors\n(rows = marker × group, green = in top-1000, gray = not)",
                 fontsize=11)
    fig.tight_layout()

    pdf_path = FIG_DIR / f"{out_stem}.pdf"
    png_path = FIG_DIR / f"{out_stem}.png"
    fig.savefig(pdf_path, dpi=150, bbox_inches="tight")
    fig.savefig(png_path, dpi=150, bbox_inches="tight")
    print(f"  saved {pdf_path}")
    print(f"  saved {png_path}")
    plt.show()
    plt.close(fig)


plot_biomarker_selection_heatmap(biomarker_df, "biomarker_selection_matrix")

  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/hvg_flavor_results_outputs/figures/biomarker_selection_matrix.pdf
  saved /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/hvg_flavor_results_outputs/figures/biomarker_selection_matrix.png


## 8. Final summary table for the meeting

A compact one-page Markdown that the user can paste into slides or notes. Aggregates current results, current outstanding cells, and the headline R²/MMD numbers.

In [21]:
summary_md = []
summary_md.append("# 5-Flavor HVG Validation — current snapshot")
summary_md.append("")
summary_md.append(f"- Eval cells available: {n_evaluated}/{n_total}")
summary_md.append(f"- Trained but not yet evaluated: {(status_df['trained'].eq('done').sum()) - n_evaluated}")
summary_md.append(f"- Not yet trained: {(status_df['trained'] != 'done').sum()}")
summary_md.append("")

if not r2_pivot.empty:
    summary_md.append("## R² of means (latent space)")
    summary_md.append("")
    summary_md.append("```")
    summary_md.append(r2_pivot.round(4).to_string())
    summary_md.append("```")
    summary_md.append("")

if not mmd_pivot.empty:
    summary_md.append("## MMD (latent space, lower is better)")
    summary_md.append("")
    summary_md.append("```")
    summary_md.append(mmd_pivot.round(4).to_string())
    summary_md.append("```")
    summary_md.append("")

if not r2_pivot.empty and "impact_cellot" in r2_pivot.columns and r2_pivot["impact_cellot"].notna().any():
    impact_only = r2_pivot["impact_cellot"].dropna().sort_values(ascending=False)
    summary_md.append("## Top-5 IMPACT_CellOT cells by R²")
    summary_md.append("")
    summary_md.append("| flavor | group | mode | R² of means |")
    summary_md.append("|---|---|---|---|")
    for (flavor, gk, mode), v in impact_only.head(5).items():
        summary_md.append(f"| `{flavor}` | {gk.upper()} | {mode.upper()} | {v:.4f} |")
    summary_md.append("")

summary_path = OUT_DIR / "summary.md"
summary_path.write_text("\n".join(summary_md))
print(f"wrote {summary_path}")
print()
print("=" * 70)
print("\n".join(summary_md))

wrote /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/hvg_flavor_results_outputs/summary.md

# 5-Flavor HVG Validation — current snapshot

- Eval cells available: 80/80
- Trained but not yet evaluated: 0
- Not yet trained: 0

## R² of means (latent space)

```
model                          scgen  impact_cellot
flavor            group mode                       
cell_ranger       a     iid   0.6782         0.7373
                        ood   0.7168         0.7632
                  b     iid   0.6950         0.8377
                        ood   0.5731         0.5038
                  c     iid   0.7189         0.8575
                        ood   0.7209         0.6155
                  d     iid   0.6891         0.7940
                        ood   0.8061         0.8377
pearson_residuals a     iid   0.6116         0.7610
                        ood   0.6783         0.7829
                  b     iid   0.7025         0.8441
                        ood   0.7276